### **Submissão 3-B — Melhor LLM Solo**

**Grupo 1 · MIA · Aprendizagem Profunda**

**▶ CORRER DEPOIS** do Notebook A (`subm3-g1-MIA-A.ipynb`)

Carrega os checkpoints já gerados pelo Notebook A — **zero chamadas API**.
Extrai as previsões do melhor modelo solo (determinado no Ensemble.ipynb).

Output: `subm3-g1-MIA-B.csv`

In [ ]:
import pandas as pd
import os
import json
from collections import Counter

LABELS = ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI']

### **1. Carregar configuração e identificar melhor modelo**

In [ ]:
with open('../Subm3/ensemble-config.json', 'r') as f:
    config = json.load(f)

best_solo = config['best_solo_model']

print(f'📊 Melhor modelo solo: {best_solo.upper()}')
print(f'   Accuracy (validação): {config["best_solo_accuracy"]:.2%}')
print(f'   Model ID: {config["models"][best_solo]["model_id"]}')

### **2. Carregar previsões do checkpoint**

In [ ]:
# Carregar checkpoint do melhor modelo (gerado no Notebook A)
ckpt_path = f'../Subm3/subm3-preds-{best_solo}.json'

if not os.path.exists(ckpt_path):
    raise FileNotFoundError(
        f'Checkpoint não encontrado: {ckpt_path}\n'
        f'Corre primeiro o Notebook A (subm3-g1-MIA-A.ipynb)!'
    )

with open(ckpt_path, 'r') as f:
    solo_preds = json.load(f)

n_valid = sum(1 for p in solo_preds if p is not None)
n_none  = sum(1 for p in solo_preds if p is None)
print(f'✅ Checkpoint carregado: {n_valid}/{len(solo_preds)} previsões válidas')
if n_none > 0:
    print(f'⚠️ {n_none} falhas — serão substituídas por "Human"')

### **3. Carregar dataset e exportar submissão B**

In [ ]:
df_subm = pd.read_csv('../database/dataset-subm3.csv', sep=';')
df_subm.columns = df_subm.columns.str.strip().str.lower()
ids = df_subm['id'].tolist()

# Substituir falhas por 'Human'
labels_b = [p if p else 'Human' for p in solo_preds]

df_b = pd.DataFrame({'ID': ids, 'Label': labels_b})
path_b = '../Subm3/subm3-g1-MIA-B.csv'
df_b.to_csv(path_b, sep=';', index=False, encoding='utf-8')

print(f'{"═" * 50}')
print(f'SUBMISSÃO B — {best_solo.upper()} (solo)')
print(f'Acc validação: {config["best_solo_accuracy"]:.2%}')
print(f'{"═" * 50}')
print(df_b['Label'].value_counts().to_string())
print(f'\n✅ {path_b} ({len(df_b)} linhas)')

### **4. Validação**

In [ ]:
assert len(df_b) == len(df_subm), 'Subm B: tamanho errado'
assert list(df_b.columns) == ['ID', 'Label'], 'Subm B: colunas erradas'
assert all(l in LABELS for l in df_b['Label']), 'Subm B: labels inválidos'
print(f'✅ Subm B OK: {len(df_b)} linhas, labels válidos')
print(f'\n🎯 Ficheiro: {path_b}')